# Train the plate detector on a free Colab GPU (disconnect-proof)

Trains a YOLO plate detector on the Roboflow **barbell-detection-yolov5** set (CC BY 4.0, ~4.1k
labelled images). **Weights save to your Google Drive every epoch**, so a dropped connection or a
PC sleep never loses progress — just re-run and it **resumes** where it died.

**Do NOT train on your PC.** Sustained GPU load froze it before. Training lives in the cloud.
**Runtime → Change runtime type → GPU (T4).** Keep the PC awake (Power → Sleep → Never).


## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Install

In [ ]:
!pip -q install ultralytics roboflow

## 3. Mount Google Drive (the disconnect-proofing)

A popup asks you to authorise Drive — click through it. Weights then write to
`MyDrive/powerlifting_runs`, which survives any disconnect.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_RUNS = "/content/drive/MyDrive/powerlifting_runs"
Path(DRIVE_RUNS).mkdir(parents=True, exist_ok=True)
print("checkpoints ->", DRIVE_RUNS)

## 4. Get the dataset (no upload)

Add your key as a **Colab Secret**: left sidebar **🔑** → **+ Add new secret** → name
`ROBOFLOW_API_KEY`, paste your Private API key, toggle **Notebook access** on. Then run.

In [ ]:
from roboflow import Roboflow
from google.colab import userdata
from pathlib import Path

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
rf.workspace("computer-vision-1bdzc").project("barbell-detection-yolov5").version(6).download(
    "yolov11", location="/content/ds"
)

# collapse every label to one class 'plate' (we only care: is there a plate?)
root = Path("/content/ds")
for t in root.rglob("*.txt"):
    if "labels" not in t.parts:
        continue
    out = []
    for ln in t.read_text().splitlines():
        p = ln.split()
        if p:
            p[0] = "0"
            out.append(" ".join(p))
    t.write_text("\n".join(out) + ("\n" if out else ""))

(root / "data.yaml").write_text(
    "path: /content/ds\ntrain: train/images\nval: valid/images\ntest: test/images\nnc: 1\nnames: ['plate']\n"
)
print("dataset ready at /content/ds")

## 5. Train (~30–60 min on a T4)

If a previous run left a checkpoint in Drive, this **resumes** it; otherwise it starts fresh.
Re-running after a disconnect just continues — no lost epochs.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

last = Path(DRIVE_RUNS) / "plate" / "weights" / "last.pt"
if last.exists():
    print("resuming from", last)
    YOLO(str(last)).train(resume=True)
else:
    YOLO("yolo11n.pt").train(
        data="/content/ds/data.yaml",
        epochs=40, imgsz=640, batch=16,
        hsv_h=0.5, hsv_s=0.7, hsv_v=0.4,   # strong hue jitter -> colour-robust (any plate colour)
        fliplr=0.5, degrees=5.0,
        device=0, project=DRIVE_RUNS, name="plate", exist_ok=True,
    )
print("best weights:", Path(DRIVE_RUNS) / "plate" / "weights" / "best.pt")

## 6. Score + download `best.pt` (also already saved in Drive)

In [ ]:
from ultralytics import YOLO
from google.colab import files
from pathlib import Path

best = str(Path(DRIVE_RUNS) / "plate" / "weights" / "best.pt")
m = YOLO(best)
metrics = m.val(data="/content/ds/data.yaml")
print("mAP@50:", round(float(metrics.box.map50), 3))   # ~0.9+ is healthy for plates
files.download(best)   # convenience copy; the real one is safe in Drive either way

## 7. Next

`best.pt` is in **MyDrive/powerlifting_runs/plate/weights/** (and your Downloads). Put it in the
repo as `models/plate.pt`, then:
- `python cli.py input/clip.mov --lift squat --plate yolo`
- Compare trained vs HSV on your eval clips — keep whichever wins.

Attribution: dataset **CC BY 4.0**, `computer-vision-1bdzc/barbell-detection-yolov5` v6.
